<font size=14>Student Mission Control Expedition 2: Orientation</font>

Before you start:
1. Make a copy of this notebook. Give it the name "SMC Expedition 2: Orientation (YOUR NAME v23.07.18)".
2. Click the Table of Contents button in the top left so that you can navigate this expedition.

## A. Introduction

Welcome back. In Expedition 1, we looked at the **altitude** of the ISS.

In this Expedition, we will look at the **orientation** of the ISS, i.e. which way it is pointed.

At first, it might seem like orientation doesn't matter. Unlike an airplane, which needs to point forwards to fly, the ISS can in principle point anywhere and stay up.

One reason is the cupola window. Below is a picture of the Sahara desert as seen from the ISS. You definitely want this window pointed at the Earth so you can observe the Earth below.

![View of the Sahara from the Cupola Window on the iSS](https://upload.wikimedia.org/wikipedia/commons/c/c2/Cupola_ISS_open_shutters_middle_crop.jpg)

We'll see another reason later. Let's dig into the data and see what we can learn.

***

## B. Getting Our Data and Importing our Code

As in Expedition 1, run the cell below, which loads all the data and code we need. You should see a message that looks something like:

```
Cloning into 'smcLab'...
...
Unpacking objects: 100% (14/14), 15.54 MiB | 4.52 MiB/s, done.
```

In [ ]:
!git clone https://github.com/joshhug/smcLab.git
!rm -rf data
!rm -rf smclib
!mv smcLab/smclib ./
!mv smcLab/data ./
!rm -rf smcLab

Now run the cell below, which loads various pieces of code we'll need later.

In [ ]:
import pandas as pd
import plotly.express as px
import re
import seaborn as sns
from smclib.utils import read_iss_data
from smclib.utils import plot_ypr
from smclib.utils import box_plot, line_plot, histogram, hourly_average

***

## C. Counting Degrees of Freedom

Before we can work with the data, we need to introduce the important concepts of **yaw**, **pitch** and **roll**.

With altitude, things were simple. There's just a single number representing how high above the Earth you are. With altitude, there is only one "degree of freedom": up or down.

By contrast, orientation is not just a number. Consider your head. How many different "degrees of freedom" do you have to move your head?

I'll observe that there are two obvious ones:

1. You can turn your head to the left or right.
2. You can look up or down.

### C.1: Task 1: The Third Degree of Rotational Freedom

Above, I pointed out two degrees of freedom for orienting your head (looking left to right, and looking up and down).

Discuss with others around you and try to think of the third. Give your answer at [this link](https://forms.gle/X47SKjXNCU9iyTT88).

These three degrees of freedom are often called the yaw, pitch, and roll, as illustrated in the image below, courtesy of [youtube user SoftwarePole](https://www.youtube.com/watch?v=pQ24NtnaLl8).

![Yaw, Pitch, and Roll](https://drive.google.com/uc?id=1i05nkyDu1hRP9U1cRn-B2O9LquNQWEHk)

### C.2 Task 2: Identifying the Last Two Degrees of Freedom

So far, we've seen the ISS has four degrees of freedom for movement:

1. Altitude (Position)
2. Yaw (Orientation)
3. Pitch (Orientation)
4. Roll (Orientation)

There are two more degrees of freedom. Discuss again to try figure out what they are. Give your answer at [this link](https://forms.gle/11wWcRh4DtgDXwRQA).



These six degrees of freedom also apply to your body, or indeed any rigid object in 3 dimensional space.

The degrees of freedom corresponding to position are often called "[translational](https://en.wikipedia.org/wiki/Translation_(geometry)".

For more:
 * Read: [https://en.wikipedia.org/wiki/Six_degrees_of_freedom](https://en.wikipedia.org/wiki/Six_degrees_of_freedom).
 * Bonus exercise: In 2 dimensional space, how many translational and rotational degrees of freedom are there? How about 4 dimensional space? For answers see [https://en.wikipedia.org/wiki/Orientation_(geometry)](https://en.wikipedia.org/wiki/Orientation_(geometry)

***

## D. Reading in the Yaw, Pitch, and Roll Data

Let's read in the historical data that contains the yaw, pitch, and roll (YPR) for particular dates spanning from November 10, 2021 to November 12, 2021.

All values are given in units of degrees, e.g.:
 * 0 degrees of pitch means pointing straight forward.
 * 90 degrees of pitch would mean pointing straight up.

As a warmup, optionally try to answer the same questions as we did for the altitude data based on the preview of the YPR data shown by Pandas:

- How many rows are in the dataset?
- On average, how many observations (rows) are recorded per hour?
- What seems to be a reasonable range for the yaw, pitch, and roll?

In [ ]:
ypr = read_iss_data("data/ypr.csv")
ypr

As before, we can use the describe function to see sumamrizes of yaw, pitch, and roll values over the timeframe provided.

In [ ]:
ypr.describe()

***

## E. Creating and Box Plotting the Hourly Average

Below, we'd like to make a box plot. Unfortunately, there's so much data (over 600,000) points that the plot will be very slow if we plot the data exactly as it is.

Instead, we'll use a technique known as resampling that will create a smaller dataset where each data point is the average of all the data for a given hour. Run the cell below and we'll explain in a bit more detail.

In [ ]:
ypr_hourly_average = ypr.resample('60min', on='dtime').mean()
ypr_hourly_average

Notice there are now only 16,415 rows, each spaced an hour apart. For example, the last row gives the yaw, pitch, and roll at `2022-11-14 22:00:00`. The values given are the averages of all the values that occurred with an hour of this time.

Some hours have no data, and thus are given values of NaN, meaning "Not a Numbeer".

Let's make a box plot of this data.

In [ ]:
box_plot(ypr_hourly_average, ["yaw", "pitch", "roll"])

### E.1: Task 3: Interpreting the Box Plot

Answer the following questions:
 * What are the median yaw, pitch, and roll values?
 * What does it mean that the box in the box plot is so tiny?
 * Why do you think the medians are non-zero?

Give your answers at [this link](https://forms.gle/EDrveAmTmbBrmymZ7).

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### E.2. Task 4: More Box Plot Interpretation

As you saw above, the median yaw, pitch, and roll values are not zero.

This is not just a data glitch. Instead, the space station is very intentionally pointing a certain direction.

The reason is to save fuel.

How could this possibly save fuel? Discuss and give your answer here: [this link](https://forms.gle/Mzg1QE6NunfxxVjPA).

### E.3. Maintaining Altitude

One answer to the question from E.2. is that the space station is oriented to **minimize drag and thus preserve altitude**.

As we discussed in expedition 1, if you go too low, you'll crash into the atmosphere and disintegrate into bajillions of tiny pieces. No fun. Even at 400 kilometers above the surface, the thin atmopshere still needs to be taken into account.

The orientation of the ISS affects the amount of drag. This is similar to when you stick your hand out the window of a moving car. If your hand is horizontal, you don't feel much resistance. But if you put your hand up vertically palm facing out towards the "wind" you'll suddenly feel much more resistance.

The solar panels of the ISS are like giant hands. If they are hitting the cross section of the wind, there will be much more drag. This is a little tricky, because in the case of the solar panels, they are also the source of power, so the engineers who maintain the space station have to minimize drag while also ensuring that the panels are generating sufficient power.

### E.4. Maintaining Orientation

The other answer is **balancing out the rotational forces from the atmosphere with the rotational forces from the difference in gravity between the top and bottom of the space station.** I'm guessing you didn't think of that one.

Without going into the details, there's a special orientation called the "torque equilibrium attitude" (attitude is another term for orientation) that minimizes the amount of rotational force experienced by the ISS.

Less force means less fuel wasted keeping the ISS oriented.

If you're curious, there's a great [youtube video here](https://www.youtube.com/watch?v=4aF7zwhlDDU) that explains the torque equilibrium attitude in more detail.

***

## F. Line Plot of Yaw, Pitch, and Roll vs. Time in November 2021

Below, we show a plot of the yaw pitch and roll between the dates of November 7th, 2021 and November 10th, 2021. We'll call this a YPR plot.

In [ ]:
plot_ypr(ypr, lower_date = '2021-11-07', upper_date = '2021-11-10')

### Task 5: Interpreting the YPR Line Plot

Answer the following question: Does the space stations seem like its orientation is relatively stable?

 Give your answer at [this link](https://forms.gle/MLuhHPprufAZkSQs5).

***

## G. Line Plot of YPR Between October 2021 and November 2022

Below, we plot a wider range over the entire date range available. The gaps are times when no data is available.

In [ ]:
plot_ypr(ypr)

You'll see the orientation of the ISS is relatively stable, but with occasional huge spikes in yaw, pitch, and roll.

These huge spikes correspond to moments when the ISS was rotating around.

Imagine standing still for a few minutes, then doing a cartwheel, then standing still again for a few minutes. Your YPR plot would look similar to the one above.

### Task 6 Investigating YPR Spikes using Lower Date and Upper Date

Let's investigate some of the spikes to see why they happened.

Run the line of code below, which will show the data only between November 11, 2021 and December 12, 2021.

Note: You could also use the Zoom feature, but this code ensures that you're looking at the exact right data.

In [ ]:
plot_ypr(ypr, lower_date = '2021-11-22', upper_date = '2021-12-12')

The ISS isn't just randomly rotating around on these days. For some reason or another, the ISS flight controllers decided to rotate the station around, and rather quickly on these specific days.

It's time to use your internet detective skills to uncover why.

As a class, work together to use Google (or another search engine) to try to find out what special things happened on the ISS on one or more of these three days.

Give your answers at [this link](https://forms.gle/W5k221GVxe8seu89A).

***

# New Section

## H. The LOAC Event

As we've seen throughout expeditions 1 and 2, the ISS must constantly work to maintain its altitude and orientation (also known as "attitude", though I use orientation to avoid confusion with altitude).

Below, we see a plot of a rather unintentional change in orientation back in July 2021.

In [ ]:
loac = read_iss_data("data/loac.csv")
plot_ypr(loac)

On that day, a new module called Nauka was adding to the space station. A few hours after docking, due to a software glitch, the Nauka's built-in thrusters automatically fired, burning until they ran out of fuel. This caused the ISS to make one and a half full rotations before being brought under control.

This event was known as the "Loss of Attitude Control" or LOAC.

Though nobody was in danger, this was a serious emergency on the space station. Losing  control of attitude had never happened before.

For more see [https://www.engineering.com/story/the-nauka-module-iss-mishap-what-happened](https://www.engineering.com/story/the-nauka-module-iss-mishap-what-happened).

### Expedition 2B: LOAC Simulation Exercise

Above, you got a chance to see how the orientations and altitude of the space station changes over time, and also saw how the Nauka's rockets mistakenly firing in July 2021 caused the space station to start spinning out of control.

To get more intuition for how objects rotate in space, and how we can burn fuel to adjust their rotation, we've created a simulation game. In this game, you'll try to write code that stops the rotation of an out of control ISS.

Note that in this simulation, for simplicity, we're only showing one degree of freedom for rotation.

To complete this programming challenge in Snap*!*, head to https://snap.berkeley.edu/snap/snap.html#present:Username=dan%20garcia&ProjectName=ISS%20LOAC%20sim

(Note for teachers: We could in principle make a Python version of this game, but it would take a lot of work)

(Note for teachers: We'll have a better landing page for this exercise later)